# Oversquashing-guided lifting — experiment grid

Self-contained Kaggle runner: it clones the repository at a **pinned branch**, runs the
grid `datasets × proxies × gamma × seeds`, measures oversquashing **before and after** the
lifting on every run, and zips the results for download. Nothing has to be edited after
upload except the config cell below.

What the grid is testing:

* the **synthetic bottleneck arm** is where an OSq-guided lifting is *supposed* to win — it
  is the falsification core, and it is never dropped by the runtime guard;
* the five TU benchmarks test that nothing is *lost* elsewhere;
* `gamma = 0` never calls the proxy, so it reproduces the proxy-free objective exactly —
  that equality is asserted below rather than assumed;
* every proxy is compared under one objective, so "which proxy wins" (including a clean
  negative result) is itself an outcome.

Analysis happens locally on `results/runs.csv`; the plots at the end are sanity checks only.

In [ ]:
# =====================================================================
# CONFIG — every knob of this notebook lives in this cell.
# =====================================================================
REPO_URL = "https://github.com/AlGoRythm3000/Differentiable-Motif-Discovery.git"
BRANCH   = "feat/osq-proxy"        # pinned on purpose: results stay attributable
REPO_DIR = "/kaggle/working/repo"
OUT_DIR  = "/kaggle/working/results"

# --- grid axes -------------------------------------------------------
DATASETS = ["synthetic_bottleneck", "MUTAG", "PROTEINS", "IMDB-BINARY", "ENZYMES", "NCI1"]
PROXIES  = ["none", "r_bar", "lambda2", "efc", "cf_bc_efc"]
GAMMAS   = [0.0, 0.01, 0.1, 1.0]   # weight of the OSq term
SEEDS    = [0, 1, 2]               # three minimum: results are reported mean +- std

# --- optimization ----------------------------------------------------
EPOCHS           = 200
PATIENCE         = 30              # early stopping on validation accuracy
LR               = 0.005
WEIGHT_DECAY     = 5e-4
BATCH_SIZE       = 32
HIDDEN_DIM       = 64
TOP_K            = 4
SPARSITY_WEIGHT  = 0.05            # never 0 with a positive gamma: the OSq term alone
                                   # is minimized by the complete graph
ENCODER          = "gcn"

# --- OSq estimator ---------------------------------------------------
HUTCH_K     = 16                   # probes / CG right-hand sides
CG_TOL      = 1e-5
CG_MAXITER  = 100
OSQ_EPS     = 1e-4                 # Laplacian grounding: keeps a disconnected structure
                                   # finite and caps how bad a bottleneck may score
OSQ_SAMPLE_GRAPHS = 8              # test graphs the before/after measurement averages over

# --- synthetic arm ---------------------------------------------------
SYNTHETIC_FAMILY  = "tree_neighbors_match"   # or "path_of_cliques_match"
SYNTHETIC_GRAPHS  = 600
SYNTHETIC_CLASSES = 4
SYNTHETIC_DEPTH   = 3

# --- runtime guard ---------------------------------------------------
# If the plan does not fit, runs are cut in this order: largest gamma first, then
# NCI1, then ENZYMES. Never the synthetic arm, never a seed.
TIME_BUDGET_S = 8.0 * 3600
SKIP_REDUNDANT_ZERO_GAMMA = True   # (proxy, gamma=0) is the same run for every proxy
INSTALL_DEPS  = True               # set False if the Kaggle image already has PyG

In [ ]:
# =====================================================================
# SETUP — dependencies, clone at the pinned branch, import
# =====================================================================
import os, subprocess, sys

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "networkx"], check=False)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("branch:", BRANCH)
print("commit:", COMMIT_SHA)   # recorded in every single result row

In [ ]:
# =====================================================================
# ENVIRONMENT LOG
# =====================================================================
import torch

from tools.experiment_grid import GridConfig, build_plan, environment_info, run_grid
from tools.results_store import ResultsStore

ENV = environment_info(REPO_DIR)
ENV["branch"] = BRANCH
for key, value in ENV.items():
    print(f"{key:>18}: {value}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{'device':>18}: {DEVICE}")

In [ ]:
# =====================================================================
# PLAN — build it and look at it before spending a session on it
# =====================================================================
config = GridConfig(
    datasets=DATASETS, proxies=PROXIES, gammas=GAMMAS, seeds=SEEDS,
    epochs=EPOCHS, patience=PATIENCE, lr=LR, weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE, hidden_dim=HIDDEN_DIM, top_k=TOP_K,
    sparsity_weight=SPARSITY_WEIGHT, encoder=ENCODER,
    hutch_k=HUTCH_K, cg_tol=CG_TOL, cg_maxiter=CG_MAXITER, osq_eps=OSQ_EPS,
    osq_sample_graphs=OSQ_SAMPLE_GRAPHS,
    synthetic_family=SYNTHETIC_FAMILY, synthetic_graphs=SYNTHETIC_GRAPHS,
    synthetic_classes=SYNTHETIC_CLASSES, synthetic_depth=SYNTHETIC_DEPTH,
    skip_redundant_zero_gamma=SKIP_REDUNDANT_ZERO_GAMMA,
    time_budget_s=TIME_BUDGET_S, device=DEVICE, data_root="/kaggle/working/datasets",
    commit_sha=COMMIT_SHA,
)

plan = build_plan(config)
print(f"{len(plan)} runs planned\n")
for spec in plan[:10]:
    print(" ", spec.run_id)
print("  ...")

store = ResultsStore(OUT_DIR)
store.write_env(ENV)

In [ ]:
# =====================================================================
# SANITY CHECK — gamma = 0 must reproduce the proxy-free objective exactly
# =====================================================================
# Cheap, and it is the one equality the whole grid rests on: if it ever fails,
# the `none` baseline is not comparable to the other arms and no result below
# means anything.
import torch

from tools.losses import DMDLoss
from tools.osq_proxies import available_proxies, get_proxy

_logits = torch.randn(6, 3)
_target = torch.randint(0, 3, (6,))
_mask = torch.ones(6, dtype=torch.bool)
_structure = {"alpha": torch.rand(6)}

_plain = DMDLoss(sparsity_weight=SPARSITY_WEIGHT)(_logits, _target, _mask, _structure)
for _name in available_proxies():
    _gated = DMDLoss(sparsity_weight=SPARSITY_WEIGHT, osq_weight=0.0,
                     osq_fn=get_proxy(_name))(_logits, _target, _mask, _structure)
    assert _gated.total.item() == _plain.total.item(), _name
print("gamma = 0 equivalence holds for:", ", ".join(available_proxies()))

In [ ]:
# =====================================================================
# RUN THE GRID
# =====================================================================
# Every run is wrapped: a failure is stored with its traceback in the row and the
# loop continues. Results are written as each run finishes, so a session that
# dies still leaves everything already completed - and re-running this cell
# resumes instead of redoing (an existing run_id is skipped, never overwritten).
summary = run_grid(config, store)

print("\n" + "=" * 60)
print(f"completed : {summary['completed']}")
print(f"failed    : {summary['failed']}")
print(f"skipped   : {summary['skipped']} (already stored)")
print(f"dropped   : {len(summary['dropped'])} (time budget)")
print(f"elapsed   : {summary['elapsed_s'] / 60:.1f} min")
if summary["dropped"]:
    print("\nruns the budget forced out:")
    for run_id in summary["dropped"][:20]:
        print("  ", run_id)

In [ ]:
# =====================================================================
# SAVE + ZIP for one-click download
# =====================================================================
archive = store.zip("/kaggle/working/osq_results.zip")
print("archive:", archive)
print("rows   :", len(store.read_runs()))
print("files  :", sorted(p.name for p in store.out_dir.iterdir()))

In [ ]:
# =====================================================================
# QUICK SANITY PLOTS (convenience only - real analysis happens locally)
# =====================================================================
import matplotlib.pyplot as plt
import pandas as pd

runs = pd.read_csv(store.runs_path)
runs = runs[runs["status"] == "ok"].copy()
for column in ["test_acc", "gamma", "r_bar_before", "r_bar_after"]:
    runs[column] = pd.to_numeric(runs[column], errors="coerce")

# Mean +- std over seeds. A single-seed number is never reported.
table = (runs.groupby(["dataset", "proxy", "gamma"])["test_acc"]
              .agg(["mean", "std", "count"]).reset_index())
display(table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for dataset, group in runs.groupby("dataset"):
    by_proxy = group.groupby("proxy")["test_acc"].mean()
    axes[0].plot(by_proxy.index, by_proxy.values, marker="o", label=dataset)
axes[0].set_ylabel("test accuracy (mean over seeds)")
axes[0].set_title("accuracy by proxy")
axes[0].legend(fontsize=7)
axes[0].tick_params(axis="x", rotation=30)

# The claim of the phase, in one picture: did the lifting actually reduce the
# measured mean effective resistance? Points below the diagonal did.
axes[1].scatter(runs["r_bar_before"], runs["r_bar_after"], c=runs["gamma"], cmap="viridis", s=18)
limit = float(runs[["r_bar_before", "r_bar_after"]].max().max())
axes[1].plot([0, limit], [0, limit], "k--", lw=1)
axes[1].set_xlabel("R_bar before lifting")
axes[1].set_ylabel("R_bar after lifting")
axes[1].set_title("measured oversquashing, before vs after")

plt.tight_layout()
plt.show()

In [ ]:
# Loss curves for a handful of runs, as a training-health check.
epochs = pd.read_csv(store.epochs_path)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for run_id, group in list(epochs.groupby("run_id"))[:8]:
    axes[0].plot(group["epoch"], group["train_loss"], lw=1, label=run_id)
    axes[1].plot(group["epoch"], group["val_acc"], lw=1)
axes[0].set_title("train loss")
axes[0].set_xlabel("epoch")
axes[0].legend(fontsize=6)
axes[1].set_title("validation accuracy")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()